# 🛡️ Hybrid Phishing Detection Pipeline

**Architecture:** Multi-stage cascade — Character-Level CNN → Light URL Feature DNN → Deep 51-Feature DNN

**Datasets:**
- `PhiUSIIL_Phishing_URL_Dataset.csv` — 235,795 rows, 54 columns, labels 0/1
- `benign_links.csv` — 1,235,654 rows, URL + label only (label=0)

> ⚠️ **Critical:** `benign_links.csv` is used ONLY for the Character-Level and Light Feature models. It must NEVER train the Deep 51-Feature model.

---
## Step 0 — Google Colab Environment Setup

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = '/content/drive/MyDrive/Hybrid_Phishing_Model/'
import os
os.makedirs(SAVE_DIR, exist_ok=True)

In [ ]:
# Install / verify dependencies
!pip install -q tensorflow keras scikit-learn joblib tqdm matplotlib seaborn

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, mixed_precision
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, f1_score
)
import joblib
from tqdm import tqdm
import gc
import re
import itertools
import warnings
warnings.filterwarnings('ignore')

# Enable mixed precision for GPU memory efficiency
mixed_precision.set_global_policy('mixed_float16')

# Reproducibility
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

# GPU check
print("GPUs:", tf.config.list_physical_devices('GPU'))
print("Mixed precision policy:", mixed_precision.global_policy())

---
## Step 1 — Data Loading and Preprocessing

### 1.1 Load PhiUSIIL Dataset

In [ ]:
PHIUSIIL_PATH = '/content/drive/MyDrive/Hybrid_Phishing_Model/PhiUSIIL_Phishing_URL_Dataset.csv'
BENIGN_PATH   = '/content/drive/MyDrive/Hybrid_Phishing_Model/benign_links.csv'

df_phi = pd.read_csv(PHIUSIIL_PATH, low_memory=False)

# Fix column name typos
rename_map = {
    'NoOfDegitsInURL':       'NoOfDigitsInURL',
    'DegitRatioInURL':       'DigitRatioInURL',
    'SpacialCharRatioInURL': 'SpecialCharRatioInURL'
}
df_phi.rename(columns=rename_map, inplace=True)

# Standardize URL column name to lowercase
df_phi.rename(columns={'URL': 'url'}, inplace=True)

# Drop non-feature columns
df_phi.drop(columns=['FILENAME', 'Domain', 'Title'], inplace=True, errors='ignore')

print(f"PhiUSIIL shape: {df_phi.shape}")
print(f"Label distribution:\n{df_phi['label'].value_counts()}")

### 1.2 Load Benign Dataset

In [ ]:
df_benign = pd.read_csv(BENIGN_PATH)
df_benign.rename(columns={'url': 'url'}, inplace=True)  # ensure lowercase
df_benign['label'] = 0

# Flag: which rows have engineered deep features
df_phi['has_deep_features']    = True
df_benign['has_deep_features'] = False

print(f"Benign dataset shape: {df_benign.shape}")

### 1.3 Merge Datasets Carefully

In [ ]:
# For benign_links: fill all 51 engineered feature columns with NaN
# They will NOT be used for Deep Model training — only for char + light models
phi_cols = [c for c in df_phi.columns if c not in ['url', 'label', 'has_deep_features']]

for col in phi_cols:
    if col not in df_benign.columns:
        df_benign[col] = np.nan

df_combined = pd.concat([df_phi, df_benign], ignore_index=True)

# Clean URL: lowercase, strip whitespace
df_combined['url'] = df_combined['url'].astype(str).str.lower().str.strip()

# Remove duplicates
df_combined.drop_duplicates(subset='url', inplace=True)

# Shuffle
df_combined = df_combined.sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f"Combined dataset shape: {df_combined.shape}")
print(f"Label distribution:\n{df_combined['label'].value_counts()}")
print(f"Rows with deep features: {df_combined['has_deep_features'].sum():,}")

### 1.4 Define Feature Groups

In [ ]:
# ── GROUP A: URL-Derivable Features (Light Model) ──────────────────────────
# These can be computed from the URL string alone — no web crawling needed
LIGHT_FEATURES = [
    'URLLength', 'DomainLength', 'IsDomainIP',
    'TLDLength', 'NoOfSubDomain',
    'HasObfuscation', 'NoOfObfuscatedChar', 'ObfuscationRatio',
    'NoOfLettersInURL', 'LetterRatioInURL',
    'NoOfDigitsInURL', 'DigitRatioInURL',
    'NoOfEqualsInURL', 'NoOfQMarkInURL', 'NoOfAmpersandInURL',
    'NoOfOtherSpecialCharsInURL', 'SpecialCharRatioInURL',
    'IsHTTPS',
    'CharContinuationRate',
    'URLCharProb',
    'URLSimilarityIndex',
    'TLDLegitimateProb',
]

# ── GROUP B: All 51 Engineered Features (Deep Model) ───────────────────────
# Includes all light features + content-dependent features (crawled from page)
DEEP_FEATURES = [
    'URLLength', 'DomainLength', 'IsDomainIP',
    'URLSimilarityIndex', 'CharContinuationRate', 'TLDLegitimateProb', 'URLCharProb',
    'TLDLength', 'NoOfSubDomain',
    'HasObfuscation', 'NoOfObfuscatedChar', 'ObfuscationRatio',
    'NoOfLettersInURL', 'LetterRatioInURL',
    'NoOfDigitsInURL', 'DigitRatioInURL',
    'NoOfEqualsInURL', 'NoOfQMarkInURL', 'NoOfAmpersandInURL',
    'NoOfOtherSpecialCharsInURL', 'SpecialCharRatioInURL',
    'IsHTTPS',
    'LineOfCode', 'LargestLineLength',
    'HasTitle', 'DomainTitleMatchScore', 'URLTitleMatchScore',
    'HasFavicon', 'Robots', 'IsResponsive',
    'NoOfURLRedirect', 'NoOfSelfRedirect',
    'HasDescription', 'NoOfPopup', 'NoOfiFrame',
    'HasExternalFormSubmit', 'HasSocialNet', 'HasSubmitButton',
    'HasHiddenFields', 'HasPasswordField',
    'Bank', 'Pay', 'Crypto', 'HasCopyrightInfo',
    'NoOfImage', 'NoOfCSS', 'NoOfJS',
    'NoOfSelfRef', 'NoOfEmptyRef', 'NoOfExternalRef',
]

# Verify all feature columns exist
assert all(f in df_combined.columns for f in LIGHT_FEATURES), "Missing light features!"
assert all(f in df_combined.columns for f in DEEP_FEATURES),  "Missing deep features!"

N_LIGHT = len(LIGHT_FEATURES)
N_DEEP  = len(DEEP_FEATURES)
print(f"Light features: {N_LIGHT}")
print(f"Deep features:  {N_DEEP}")

### 1.5 Feature Scaling

In [ ]:
# ── Scale LIGHT features (on combined dataset) ─────────────────────────────
# Use median imputation for NaN values in benign rows
df_combined[LIGHT_FEATURES] = df_combined[LIGHT_FEATURES].fillna(
    df_combined[LIGHT_FEATURES].median()
)

scaler_light = StandardScaler()
df_combined[LIGHT_FEATURES] = scaler_light.fit_transform(df_combined[LIGHT_FEATURES])

# ── Scale DEEP features (ONLY on rows with deep features) ──────────────────
df_deep_only = df_combined[df_combined['has_deep_features']].copy()
df_deep_only[DEEP_FEATURES] = df_deep_only[DEEP_FEATURES].fillna(
    df_deep_only[DEEP_FEATURES].median()
)

scaler_deep = StandardScaler()
df_deep_only[DEEP_FEATURES] = scaler_deep.fit_transform(df_deep_only[DEEP_FEATURES])

# Save scalers
joblib.dump(scaler_light, os.path.join(SAVE_DIR, 'scaler_light.pkl'))
joblib.dump(scaler_deep,  os.path.join(SAVE_DIR, 'scaler_deep.pkl'))
print("Scalers saved.")

### 1.6 Character-Level Tokenization

In [ ]:
MAX_URL_LENGTH = 200  # covers 99%+ of URLs in dataset

# Build character vocabulary from ALL URLs (combined dataset)
all_chars  = set(''.join(df_combined['url'].tolist()))
char_vocab = {char: idx + 1 for idx, char in enumerate(sorted(all_chars))}
char_vocab['<PAD>'] = 0
VOCAB_SIZE = len(char_vocab)

def url_to_sequence(url, vocab, max_len):
    seq = [vocab.get(c, 0) for c in url[:max_len]]
    seq = seq + [0] * (max_len - len(seq))
    return seq

# Save character artifacts
joblib.dump(char_vocab,     os.path.join(SAVE_DIR, 'char_vocab.pkl'))
joblib.dump(MAX_URL_LENGTH, os.path.join(SAVE_DIR, 'max_length.pkl'))
print(f"Vocab size: {VOCAB_SIZE}, Max URL length: {MAX_URL_LENGTH}")

---
## Step 2 — Prepare Training Splits

### 2.1 Character + Light Model Split (full combined dataset)

In [ ]:
X_char  = np.array([url_to_sequence(u, char_vocab, MAX_URL_LENGTH)
                    for u in tqdm(df_combined['url'], desc="Tokenizing")])
X_light = df_combined[LIGHT_FEATURES].values.astype(np.float32)
y_all   = df_combined['label'].values.astype(np.float32)

(
    X_char_train, X_char_test,
    X_light_train, X_light_test,
    y_train, y_test
) = train_test_split(
    X_char, X_light, y_all,
    test_size=0.2, random_state=SEED, stratify=y_all
)

print(f"Train: {len(y_train):,}  |  Test: {len(y_test):,}")
print(f"Train phishing rate: {y_train.mean():.3f}")

### 2.2 Deep Feature Model Split (PhiUSIIL rows only)

In [ ]:
X_deep_all = df_deep_only[DEEP_FEATURES].values.astype(np.float32)
y_deep_all = df_deep_only['label'].values.astype(np.float32)

X_deep_train, X_deep_test, y_deep_train, y_deep_test = train_test_split(
    X_deep_all, y_deep_all,
    test_size=0.2, random_state=SEED, stratify=y_deep_all
)

print(f"Deep train: {len(y_deep_train):,}  |  Deep test: {len(y_deep_test):,}")

---
## Step 3 — Model Definitions

### 3.1 Character-Level CNN Model

In [ ]:
def build_char_model(vocab_size, max_len, embed_dim=128):
    inputs = keras.Input(shape=(max_len,), name='char_input')

    x = layers.Embedding(input_dim=vocab_size + 1, output_dim=embed_dim,
                         input_length=max_len, name='char_embedding')(inputs)

    x = layers.Conv1D(256, kernel_size=7, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(pool_size=3)(x)

    x = layers.Conv1D(128, kernel_size=5, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(pool_size=3)(x)

    x = layers.Conv1D(64, kernel_size=3, padding='same', activation='relu')(x)
    x = layers.GlobalMaxPooling1D()(x)

    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.4)(x)

    outputs = layers.Dense(1, activation='sigmoid', dtype='float32', name='char_output')(x)

    model = keras.Model(inputs=inputs, outputs=outputs, name='CharModel')
    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss='binary_crossentropy',
        metrics=['accuracy', keras.metrics.AUC(name='auc')]
    )
    return model

char_model = build_char_model(VOCAB_SIZE, MAX_URL_LENGTH)
char_model.summary()

### 3.2 Light URL Features Model

In [ ]:
def build_light_model(n_features):
    inputs = keras.Input(shape=(n_features,), name='light_input')

    x = layers.Dense(256)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.2)(x)

    x = layers.Dense(64, activation='relu')(x)

    outputs = layers.Dense(1, activation='sigmoid', dtype='float32', name='light_output')(x)

    model = keras.Model(inputs=inputs, outputs=outputs, name='LightModel')
    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss='binary_crossentropy',
        metrics=['accuracy', keras.metrics.AUC(name='auc')]
    )
    return model

light_model = build_light_model(N_LIGHT)
light_model.summary()

### 3.3 Deep 51-Feature Model

In [ ]:
def build_deep_model(n_features):
    inputs = keras.Input(shape=(n_features,), name='deep_input')

    x = layers.Dense(512)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.4)(x)

    x = layers.Dense(256)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.3)(x)

    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dense(64,  activation='relu')(x)

    outputs = layers.Dense(1, activation='sigmoid', dtype='float32', name='deep_output')(x)

    model = keras.Model(inputs=inputs, outputs=outputs, name='DeepModel')
    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss='binary_crossentropy',
        metrics=['accuracy', keras.metrics.AUC(name='auc')]
    )
    return model

deep_model = build_deep_model(N_DEEP)
deep_model.summary()

---
## Step 4 — Training Configuration and Callbacks

In [ ]:
def get_callbacks(model_name, save_dir):
    return [
        keras.callbacks.EarlyStopping(
            monitor='val_auc', patience=7,
            restore_best_weights=True, mode='max'
        ),
        keras.callbacks.ModelCheckpoint(
            filepath=os.path.join(save_dir, f'{model_name}.keras'),
            monitor='val_auc', save_best_only=True, mode='max',
            verbose=1
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5,
            patience=3, min_lr=1e-6, verbose=1
        ),
    ]

EPOCHS = 60
print(f"Max epochs: {EPOCHS}")

---
## Step 5 — Training (Sequential — Clear GPU Between Models)

### 5.1 Train Character-Level Model

In [ ]:
print("=" * 60)
print("TRAINING: Character-Level CNN Model")
print("=" * 60)

BATCH_SIZE_CHAR = 512   # Large batch — sequence model is memory-efficient

history_char = char_model.fit(
    X_char_train, y_train,
    validation_split=0.1,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE_CHAR,
    callbacks=get_callbacks('char_model', SAVE_DIR),
    class_weight={0: 1.0, 1: (y_train == 0).sum() / (y_train == 1).sum()},
    verbose=1
)

# Reload best weights
char_model = keras.models.load_model(os.path.join(SAVE_DIR, 'char_model.keras'))
print("Char model training complete.")

### 5.2 Clear GPU Memory, Train Light Feature Model

In [ ]:
# GPU memory cleanup between models
gc.collect()
tf.keras.backend.clear_session()

# Re-instantiate only what we need next
light_model = build_light_model(N_LIGHT)

print("=" * 60)
print("TRAINING: Light URL Features Model")
print("=" * 60)

BATCH_SIZE_LIGHT = 512

history_light = light_model.fit(
    X_light_train, y_train,
    validation_split=0.1,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE_LIGHT,
    callbacks=get_callbacks('light_feature_model', SAVE_DIR),
    class_weight={0: 1.0, 1: (y_train == 0).sum() / (y_train == 1).sum()},
    verbose=1
)

light_model = keras.models.load_model(os.path.join(SAVE_DIR, 'light_feature_model.keras'))
print("Light model training complete.")

### 5.3 Clear GPU Memory, Train Deep 51-Feature Model

In [ ]:
gc.collect()
tf.keras.backend.clear_session()

deep_model = build_deep_model(N_DEEP)

print("=" * 60)
print("TRAINING: Deep 51-Feature Model (PhiUSIIL only)")
print("=" * 60)

BATCH_SIZE_DEEP = 256
deep_class_weight = {
    0: 1.0,
    1: (y_deep_train == 0).sum() / (y_deep_train == 1).sum()
}

history_deep = deep_model.fit(
    X_deep_train, y_deep_train,
    validation_split=0.1,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE_DEEP,
    callbacks=get_callbacks('deep_feature_model', SAVE_DIR),
    class_weight=deep_class_weight,
    verbose=1
)

deep_model = keras.models.load_model(os.path.join(SAVE_DIR, 'deep_feature_model.keras'))
print("Deep model training complete.")

---
## Step 6 — Evaluation and Visualization

### 6.1 Training History Plots

In [ ]:
def plot_history(history, model_name):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    for ax, metric, ylabel in zip(
        axes,
        ['loss', 'accuracy', 'auc'],
        ['Loss', 'Accuracy', 'AUC']
    ):
        ax.plot(history.history[metric],          label='Train')
        ax.plot(history.history[f'val_{metric}'], label='Validation')
        ax.set_title(f'{model_name} — {ylabel}')
        ax.set_xlabel('Epoch')
        ax.set_ylabel(ylabel)
        ax.legend()
        ax.grid(True)

    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, f'{model_name}_history.png'), dpi=150)
    plt.show()

plot_history(history_char,  'CharModel')
plot_history(history_light, 'LightModel')
plot_history(history_deep,  'DeepModel')

### 6.2 Evaluation Function

In [ ]:
def evaluate_model(model, X_test, y_test, model_name, threshold=0.5):
    y_prob = model.predict(X_test, batch_size=512, verbose=0).ravel()
    y_pred = (y_prob >= threshold).astype(int)

    print(f"\n{'='*60}")
    print(f"EVALUATION: {model_name}")
    print('='*60)
    print(classification_report(y_test, y_pred,
                                 target_names=['Legitimate', 'Phishing']))

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Legitimate', 'Phishing'],
                yticklabels=['Legitimate', 'Phishing'], ax=axes[0])
    axes[0].set_title(f'{model_name} — Confusion Matrix')
    axes[0].set_ylabel('Actual')
    axes[0].set_xlabel('Predicted')

    # ROC Curve
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc_score   = roc_auc_score(y_test, y_prob)
    axes[1].plot(fpr, tpr, label=f'AUC = {auc_score:.4f}', color='darkorange')
    axes[1].plot([0, 1], [0, 1], 'k--')
    axes[1].set_xlabel('False Positive Rate')
    axes[1].set_ylabel('True Positive Rate')
    axes[1].set_title(f'{model_name} — ROC Curve')
    axes[1].legend()
    axes[1].grid(True)

    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, f'{model_name}_evaluation.png'), dpi=150)
    plt.show()

    return y_prob, auc_score

# Reload all models (safe to call even if session was cleared)
char_model  = keras.models.load_model(os.path.join(SAVE_DIR, 'char_model.keras'))
light_model = keras.models.load_model(os.path.join(SAVE_DIR, 'light_feature_model.keras'))
deep_model  = keras.models.load_model(os.path.join(SAVE_DIR, 'deep_feature_model.keras'))

prob_char,  auc_char  = evaluate_model(char_model,  X_char_test,  y_test,      'CharModel')
prob_light, auc_light = evaluate_model(light_model, X_light_test, y_test,      'LightModel')
prob_deep,  auc_deep  = evaluate_model(deep_model,  X_deep_test,  y_deep_test, 'DeepModel')

### 6.3 ROC-Based Threshold Tuning

In [ ]:
def find_optimal_threshold(y_true, y_prob, model_name):
    thresholds = np.arange(0.1, 0.9, 0.01)
    f1_scores  = [f1_score(y_true, (y_prob >= t).astype(int)) for t in thresholds]
    best_t     = thresholds[np.argmax(f1_scores)]
    print(f"{model_name} — Optimal threshold: {best_t:.2f} (F1={max(f1_scores):.4f})")
    return best_t

THRESHOLD_CHAR  = find_optimal_threshold(y_test,      prob_char,  'CharModel')
THRESHOLD_LIGHT = find_optimal_threshold(y_test,      prob_light, 'LightModel')
THRESHOLD_DEEP  = find_optimal_threshold(y_deep_test, prob_deep,  'DeepModel')

# Save thresholds
joblib.dump({
    'char':  THRESHOLD_CHAR,
    'light': THRESHOLD_LIGHT,
    'deep':  THRESHOLD_DEEP
}, os.path.join(SAVE_DIR, 'thresholds.pkl'))
print("Thresholds saved.")

---
## Step 7 — Inference / Testing Script

### 7.1 Feature Extraction Functions for New URLs

In [ ]:
import re
import itertools
from urllib.parse import urlparse

def extract_light_features_from_url(url: str) -> np.ndarray:
    """
    Extract the 22 light features from a raw URL string.
    Returns values in the same order as LIGHT_FEATURES.
    """
    url = url.lower().strip()
    parsed  = urlparse(url)
    domain  = parsed.netloc or ''
    tld     = domain.split('.')[-1] if '.' in domain else ''
    subdoms = domain.split('.')[:-2]  # parts before SLD + TLD

    # Simplified TLD legitimacy lookup
    TLD_PROB = {
        'com': 0.52, 'org': 0.40, 'net': 0.38, 'edu': 0.95,
        'gov': 0.98, 'io':  0.30, 'co':  0.35, 'uk':  0.60,
        'de':  0.65, 'ru':  0.20, 'xyz': 0.05, 'tk':  0.02
    }

    letters = re.findall(r'[a-z]', url)
    digits  = re.findall(r'[0-9]', url)
    special = re.findall(r'[^a-z0-9]', url)

    url_len  = len(url)
    dom_len  = len(domain)
    is_ip    = 1 if re.match(r'^\d{1,3}(\.\d{1,3}){3}$', domain) else 0
    tld_len  = len(tld)
    n_subdom = len(subdoms)

    obf_chars = re.findall(r'%[0-9a-f]{2}|@', url)
    has_obf   = 1 if obf_chars else 0
    n_obf     = len(obf_chars)
    obf_ratio = n_obf / url_len if url_len > 0 else 0

    n_letters  = len(letters)
    let_ratio  = n_letters / url_len if url_len > 0 else 0
    n_digits   = len(digits)
    dig_ratio  = n_digits / url_len if url_len > 0 else 0
    n_equals   = url.count('=')
    n_qmarks   = url.count('?')
    n_amps     = url.count('&')
    n_other    = len(re.findall(r'[^a-z0-9=?&/:.@%-]', url))
    spec_ratio = len(special) / url_len if url_len > 0 else 0
    is_https   = 1 if parsed.scheme == 'https' else 0

    # Character continuation rate: max consecutive same-char run / url_len
    max_run   = max((len(list(g)) for _, g in itertools.groupby(url)), default=0)
    char_cont = max_run / url_len if url_len > 0 else 0

    # URL character probability: ratio of alphanumeric chars
    url_char_prob = (n_letters + n_digits) / url_len if url_len > 0 else 0

    # URL similarity index: simplified lookup against known domains
    KNOWN_DOMAINS = {
        'google.com', 'microsoft.com', 'facebook.com',
        'amazon.com', 'apple.com', 'paypal.com', 'github.com'
    }
    url_sim  = 100 if any(kd in domain for kd in KNOWN_DOMAINS) else min(50, dom_len)
    tld_legit = TLD_PROB.get(tld, 0.15)

    # Return in same order as LIGHT_FEATURES list
    return np.array([
        url_len, dom_len, is_ip, url_sim, char_cont, tld_legit, url_char_prob,
        tld_len, n_subdom, has_obf, n_obf, obf_ratio,
        n_letters, let_ratio, n_digits, dig_ratio,
        n_equals, n_qmarks, n_amps, n_other, spec_ratio, is_https
    ], dtype=np.float32)

### 7.2 Main Inference Pipeline

In [ ]:
def predict_url(
    url: str,
    char_model, light_model, deep_model,
    char_vocab, scaler_light, scaler_deep,
    max_len, thresholds,
    cascade_threshold: float = 0.3,
    verbose: bool = True
) -> dict:
    """
    Multi-stage inference pipeline.

    Stage 1: Character CNN + Light Feature Model (fast, always runs)
    Stage 2: Deep 51-Feature Model (runs only if Stage 1 suspects phishing)
    """
    url_clean = url.lower().strip()

    # ── Stage 1a: Character-Level ──────────────────────────────────────────
    seq    = np.array([url_to_sequence(url_clean, char_vocab, max_len)])
    p_char = float(char_model.predict(seq, verbose=0)[0][0])

    # ── Stage 1b: Light Features ───────────────────────────────────────────
    light_raw    = extract_light_features_from_url(url_clean).reshape(1, -1)
    light_scaled = scaler_light.transform(light_raw)
    p_light      = float(light_model.predict(light_scaled, verbose=0)[0][0])

    # ── Stage 1 Decision ───────────────────────────────────────────────────
    stage1_suspicious = (p_char >= cascade_threshold) or (p_light >= cascade_threshold)
    stage1_prob       = max(p_char, p_light)

    # ── Stage 2: Deep Model (conditional) ─────────────────────────────────
    p_deep     = None
    final_prob = stage1_prob

    if stage1_suspicious:
        # NOTE: In production, extract full 51 features via web crawling.
        # For offline inference, use available light features + zero-fill content features.
        light_raw_vals = extract_light_features_from_url(url_clean)
        deep_raw = np.zeros(len(DEEP_FEATURES), dtype=np.float32)
        for i, feat in enumerate(DEEP_FEATURES):
            if feat in LIGHT_FEATURES:
                li = LIGHT_FEATURES.index(feat)
                deep_raw[i] = light_raw_vals[li]

        deep_scaled = scaler_deep.transform(deep_raw.reshape(1, -1))
        p_deep      = float(deep_model.predict(deep_scaled, verbose=0)[0][0])
        final_prob  = p_deep  # Deep model has final say

    # ── Final Decision ─────────────────────────────────────────────────────
    final_pred = 'PHISHING' if final_prob >= thresholds['deep'] else 'LEGITIMATE'

    result = {
        'url':            url,
        'p_char':         round(p_char,  4),
        'p_light':        round(p_light, 4),
        'p_deep':         round(p_deep,  4) if p_deep is not None else 'N/A (Stage 1 clean)',
        'final_prob':     round(final_prob, 4),
        'prediction':     final_pred,
        'stage2_invoked': stage1_suspicious,
    }

    if verbose:
        print(f"\n{'─'*60}")
        print(f"  URL:                {url}")
        print(f"  Char Probability:   {result['p_char']:.4f}")
        print(f"  Light Probability:  {result['p_light']:.4f}")
        print(f"  Deep Probability:   {result['p_deep']}")
        print(f"  ► Phishing Prob:    {result['final_prob']:.4f}")
        print(f"  ► Prediction:       {result['prediction']}")
        print(f"{'─'*60}")

    return result

### 7.3 Load Artifacts and Run Inference

In [ ]:
# ── Load all artifacts ─────────────────────────────────────────────────────
char_model  = keras.models.load_model(os.path.join(SAVE_DIR, 'char_model.keras'))
light_model = keras.models.load_model(os.path.join(SAVE_DIR, 'light_feature_model.keras'))
deep_model  = keras.models.load_model(os.path.join(SAVE_DIR, 'deep_feature_model.keras'))

char_vocab   = joblib.load(os.path.join(SAVE_DIR, 'char_vocab.pkl'))
max_length   = joblib.load(os.path.join(SAVE_DIR, 'max_length.pkl'))
scaler_light = joblib.load(os.path.join(SAVE_DIR, 'scaler_light.pkl'))
scaler_deep  = joblib.load(os.path.join(SAVE_DIR, 'scaler_deep.pkl'))
thresholds   = joblib.load(os.path.join(SAVE_DIR, 'thresholds.pkl'))

# ── Single URL Test ────────────────────────────────────────────────────────
TEST_URL = "http://paypal-secure-login-update.xyz"
result = predict_url(
    TEST_URL, char_model, light_model, deep_model,
    char_vocab, scaler_light, scaler_deep, max_length, thresholds
)

### 7.4 Batch URL Test and Visualization

In [ ]:
TEST_URLS = [
    "https://google.com",
    "https://microsoft.com",
    "http://paypal-secure-login-update.xyz",
    "http://192.168.1.1/bank-login",
    "https://github.com/openai/whisper",
    "http://free-iphone-giveaway-click-now.tk/win",
    "https://stackoverflow.com/questions",
    "http://amazon-security-alert-update.ru/verify",
]

results = [
    predict_url(u, char_model, light_model, deep_model,
                char_vocab, scaler_light, scaler_deep,
                max_length, thresholds)
    for u in TEST_URLS
]

# ── Visualize ─────────────────────────────────────────────────────────────
from matplotlib.patches import Patch

def plot_batch_results(results):
    urls   = [r['url'][:50] + '...' if len(r['url']) > 50 else r['url'] for r in results]
    probs  = [r['final_prob'] for r in results]
    colors = ['#d62728' if r['prediction'] == 'PHISHING' else '#2ca02c' for r in results]

    fig, ax = plt.subplots(figsize=(12, max(5, len(results) * 0.7)))
    bars = ax.barh(urls, probs, color=colors)
    ax.axvline(x=0.5, color='black', linestyle='--', alpha=0.5, label='Threshold 0.5')
    ax.set_xlabel('Phishing Probability')
    ax.set_title('Batch URL Phishing Analysis')
    ax.set_xlim(0, 1)

    for bar, prob in zip(bars, probs):
        ax.text(min(prob + 0.02, 0.95), bar.get_y() + bar.get_height() / 2,
                f'{prob:.3f}', va='center', fontsize=9)

    ax.legend(handles=[
        Patch(color='#d62728', label='Phishing'),
        Patch(color='#2ca02c', label='Legitimate')
    ])
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, 'batch_results.png'), dpi=150)
    plt.show()

plot_batch_results(results)

---
## Step 8 — Artifacts Summary

After full execution, the following files will exist in
`/content/drive/MyDrive/Hybrid_Phishing_Model/`:

| File | Description |
|------|-------------|
| `char_model.keras` | Trained Character-Level CNN |
| `light_feature_model.keras` | Trained Light Features DNN |
| `deep_feature_model.keras` | Trained Deep 51-Feature DNN |
| `scaler_light.pkl` | StandardScaler for light features |
| `scaler_deep.pkl` | StandardScaler for deep features |
| `char_vocab.pkl` | Character-to-index vocabulary |
| `max_length.pkl` | Maximum URL sequence length |
| `thresholds.pkl` | Optimal decision thresholds per model |
| `CharModel_history.png` | Training curves — Char model |
| `LightModel_history.png` | Training curves — Light model |
| `DeepModel_history.png` | Training curves — Deep model |
| `CharModel_evaluation.png` | Confusion matrix + ROC — Char model |
| `LightModel_evaluation.png` | Confusion matrix + ROC — Light model |
| `DeepModel_evaluation.png` | Confusion matrix + ROC — Deep model |
| `batch_results.png` | Batch inference visualization |

---
## ⚠️ Critical Implementation Rules

1. **NEVER** train the Deep 51-Feature Model on rows from `benign_links.csv`.
2. **ALWAYS** fix the 3 column name typos at load time before any processing.
3. **ALWAYS** clear GPU memory between model trainings: `gc.collect(); tf.keras.backend.clear_session()`
4. **ALWAYS** use `class_weight` to handle the severe imbalance between benign_links (1.2M) and PhiUSIIL phishing samples.
5. **ALWAYS** use `stratify=y` in `train_test_split` to maintain label ratios.
6. **ALWAYS** monitor `val_auc` (not `val_loss`) for EarlyStopping and ModelCheckpoint.
7. The Cascade Threshold (default 0.3) for triggering Stage 2 should be **lower than the final decision threshold** to minimize false negatives.